# Notebook 1: The Rover's Moves (linear combinations)

**Predict, then run.** Run the two setup cells first.

**Setup.** The next cell is the drawing library, `vecdraw`. It is short, and you are allowed to read it and change it.

In [ ]:
%%writefile vecdraw.py
"""
vecdraw - a tiny helper for drawing vectors on graph paper.

READ ME! Nothing in here is magic. Every function is a few lines of
NumPy and matplotlib that you could have written yourself.
Change something, run the cell again, and see what happens.
"""
import numpy as np
import matplotlib.pyplot as plt

# Colors that are easy to tell apart (also for color-blind readers).
COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]

_size = 6     # how far the graph paper reaches in each direction
_count = 0    # how many arrows we have drawn so far (picks the next color)


def new_grid(size=6):
    """Start a fresh sheet of graph paper that runs from -size to +size."""
    global _size, _count
    _size = size
    _count = 0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_aspect("equal")                  # one step right = one step up
    if size <= 12:                          # small paper: a line at every whole number
        ax.set_xticks(range(-size, size + 1))
        ax.set_yticks(range(-size, size + 1))
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="black", linewidth=1)   # the x-axis
    ax.axvline(0, color="black", linewidth=1)   # the y-axis
    return ax


def _paper():
    """Give back the sheet we are drawing on. Make one if there is none."""
    if plt.get_fignums():
        return plt.gca()
    return new_grid(_size)


def _next_color():
    """Pick the next color from the list, starting over when we run out."""
    global _count
    color = COLORS[_count % len(COLORS)]
    _count = _count + 1
    return color


def draw_vector(v, start=(0, 0), color=None, label=None, label_at="tip"):
    """Draw the vector v as an arrow that begins at start.

    The label goes just past the tip, or beside the middle of the arrow
    if you say label_at="middle".
    Gives back the tip of the arrow (start + v), so the next arrow
    can begin where this one ended.
    """
    ax = _paper()
    v = np.array(v, dtype=float)
    start = np.array(start, dtype=float)
    if color is None:
        color = _next_color()
    ax.quiver(start[0], start[1], v[0], v[1],
              angles="xy", scale_units="xy", scale=1,   # draw at true size
              color=color, width=0.008, zorder=3)
    tip = start + v
    if label is not None:
        length = np.linalg.norm(v)
        if length == 0:
            spot = tip + np.array([0.0, _size * 0.07])
        elif label_at == "middle":
            sideways = np.array([-v[1], v[0]]) / length   # v turned a quarter turn
            spot = start + v / 2 + sideways * _size * 0.07
        else:
            spot = tip + v / length * _size * 0.07        # just past the tip
        ax.text(spot[0], spot[1], label, color=color,
                fontsize=13, fontweight="bold", ha="center", va="center")
    return tip


def draw_point(p, label=None, color="black"):
    """Draw a dot at the point p, with an optional name next to it."""
    ax = _paper()
    ax.plot(p[0], p[1], "o", color=color, markersize=8, zorder=4)
    if label is not None:
        ax.text(p[0] + _size * 0.04, p[1] + _size * 0.04, label,
                color=color, fontsize=12)


def draw_path(steps, start=(0, 0), labels=None):
    """Draw a list of vectors tip-to-tail, like a trip with several legs.

    Gives back the place where the trip ends.
    """
    place = np.array(start, dtype=float)
    for i, step in enumerate(steps):
        name = labels[i] if labels is not None else None
        place = draw_vector(step, start=place, label=name, label_at="middle")
    return place


def draw_dots(points, color="gray", size=18):
    """Draw many points at once. points is a list of [x, y] pairs."""
    ax = _paper()
    points = np.array(points, dtype=float)
    ax.scatter(points[:, 0], points[:, 1], s=size, color=color, zorder=2)


def draw_skewed_grid(b1, b2, color="#E69F00"):
    """Draw the slanted graph paper you get from the two moves b1 and b2."""
    ax = _paper()
    n = 3 * _size                           # enough lines to cover the sheet
    b1 = np.array(b1, dtype=float)
    b2 = np.array(b2, dtype=float)
    for k in range(-n, n + 1):
        # a line in the b1 direction, shifted over by k copies of b2
        begin = k * b2 - n * b1
        end = k * b2 + n * b1
        ax.plot([begin[0], end[0]], [begin[1], end[1]], color=color, linewidth=1, alpha=0.7)
        # a line in the b2 direction, shifted over by k copies of b1
        begin = k * b1 - n * b2
        end = k * b1 + n * b2
        ax.plot([begin[0], end[0]], [begin[1], end[1]], color=color, linewidth=1, alpha=0.7)


def angle_between(a, b):
    """The angle between vectors a and b, in degrees."""
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    cosine = np.clip(cosine, -1, 1)     # guard against tiny rounding errors
    degrees = np.degrees(np.arccos(cosine))
    return round(float(degrees), 2)     # two decimal places is plenty


def show():
    """Show the picture."""
    plt.show()

In [ ]:
import numpy as np
import importlib, vecdraw
importlib.reload(vecdraw)      # pick up any changes you make to the library
from vecdraw import *

## Warm-up: same skills, new numbers

Do these on paper. Then run the cell to check.

1. `3 * (2, -1) + 2 * (1, 4)`
2. The length of `(8, 6)`
3. `(2, -1)` dot `(3, 6)`. Are they perpendicular?

In [ ]:
u = np.array([2, -1])
v = np.array([1, 4])
print(3 * u + 2 * v)
print(np.linalg.norm(np.array([8, 6])))
print(np.dot(u, np.array([3, 6])))

## Part 1: Two drive modes

Your rover has two drive modes.

- Mode **a** moves 1 east and 2 north every minute.
- Mode **b** moves 3 east and 1 north every minute.

Ride mode a for 2 minutes, then mode b for 1 minute. Where are you? Work it out on paper first.

In [ ]:
a = np.array([1, 2])
b = np.array([3, 1])

trip = 2 * a + 1 * b
print(trip)

`draw_path` draws the legs of a trip tip to tail and tells you where the trip ends.

In [ ]:
new_grid(8)
end = draw_path([2 * a, 1 * b], labels=["2a", "1b"])
show()
print("the trip ends at", end)

## Part 2: Reach the charger

The charger is at `(7, 4)`. How many minutes of each mode do you need?

**Solve it on graph paper first.** Then put your answer in `c` and `d`.

The word `if` asks a yes-or-no question. The indented line under it only runs when the answer is yes.

In [ ]:
charger = np.array([7, 4])

c = 0     # minutes of mode a  (change me)
d = 0     # minutes of mode b  (change me)

trip = c * a + d * b

new_grid(8)
draw_path([c * a, d * b], labels=["a part", "b part"])
draw_point(charger, "charger")
show()

if np.array_equal(trip, charger):
    print("You made it!")
else:
    print("Not yet. You are at", trip)

## Part 3: A harder charger

Move the charger to `(5, 0)` and try again in the cell above.

Stuck? Your rover has a reverse gear. What would riding mode a for **-1** minutes mean?

## Part 4: Does the order of the legs matter?

Ride a first and then b. Or ride b first and then a. Predict what the picture looks like.

In [ ]:
c = 2
d = 1
new_grid(8)
draw_path([c * a, d * b])     # a first, then b
draw_path([d * b, c * a])     # b first, then a
show()

## Part 5: Let the computer do the boring part

A `for` loop repeats the indented lines, once for each value. `range(4)` means 0, 1, 2, 3.

Predict every line this prints.

In [ ]:
for c in range(4):
    print(c, "minutes of a gets you to", c * a)

`range(-2, 3)` means -2, -1, 0, 1, 2. It stops just **before** the second number.

Suppose you already rode mode b for 2 minutes. Which amount of mode a hits the charger at `(7, 4)`?

In [ ]:
charger = np.array([7, 4])
d = 2
for c in range(-2, 3):
    trip = c * a + d * b
    print("c =", c, " d =", d, " ends at", trip)

A loop inside a loop tries **every** pair of `c` and `d`. This is a robot that does guess-and-check for you.

How many pairs does it try? Work it out before you run it.

In [ ]:
charger = np.array([7, 4])
tries = 0
for c in range(-5, 6):
    for d in range(-5, 6):
        tries = tries + 1
        trip = c * a + d * b
        if np.array_equal(trip, charger):
            print("Found it!  c =", c, " d =", d)
print("pairs tried:", tries)

**Your turn.** Change the charger to `(5, 0)`, then `(-2, 6)`, then a spot of your own. Does the robot always find an answer?

## Part 6: When whole minutes are not enough

Set the charger to `(1, 0)` in the search above. The robot finds nothing. Does that mean the rover cannot get there?

Try these amounts. (Tiny leftovers like `0.0000000001` are just the computer rounding.)

In [ ]:
c = -0.2
d = 0.4
print(c * a + d * b)

## The name for this

Any trip made by scaling your moves and adding them up is called a **linear combination** of the moves.

`c * a + d * b` is a linear combination of `a` and `b`. The numbers `c` and `d` are called the **weights**.

## Check yourself

On paper, then check with code.

1. What is `2 * (1, 2) + 3 * (3, 1)`?
2. Find `c` and `d` so that `c * (1, 2) + d * (3, 1) = (4, 3)`.
3. Find `c` and `d` so that `c * (1, 1) + d * (0, 1) = (4, 7)`.
4. Write a spot that needs a negative weight to reach with `a` and `b`.

In [ ]:
print(2 * a + 3 * b)

---
# The Deep End

## Deep 1: The far charger

The charger is at `(107, 64)`. Make the robot search from -50 to 50. How many pairs does it try?

Now imagine the charger were a million steps away. *Wouldn't it be nice to calculate the answer exactly instead of searching?*

Try it on paper. The rover must go 107 east and 64 north, so you have two clues:

- east: `1*c + 3*d = 107`
- north: `2*c + 1*d = 64`

Can you find `c` and `d` from the clues?

In [ ]:
charger = np.array([107, 64])
tries = 0
for c in range(-50, 51):
    for d in range(-50, 51):
        tries = tries + 1
        if np.array_equal(c * a + d * b, charger):
            print("Found it!  c =", c, " d =", d)
print("pairs tried:", tries)

NumPy has a tool that solves the two clues at once. For now it is a black box. Later in the course we will open the box.

*This leads to: matrices and solving systems (Modules 6 and 9).*

In [ ]:
moves = np.column_stack([a, b])     # put the two moves side by side
print(moves)
print(np.linalg.solve(moves, charger))

## Deep 2: Mixing colors

A color on a screen is a vector with **three** numbers: how much red, green and blue, each from 0 to 1.

Orange is a linear combination of red and green. Make purple. Make teal. Make your favorite color.

Which colors can you **never** make using only red and green?

*This leads to: span (Module 2).*

In [ ]:
import matplotlib.pyplot as plt

red = np.array([1, 0, 0])
green = np.array([0, 1, 0])
blue = np.array([0, 0, 1])

color = 1.0 * red + 0.5 * green + 0.0 * blue
print(color)

plt.figure(figsize=(2, 2))
plt.imshow([[color]])        # one big pixel
plt.axis("off")
plt.show()

## Deep 3: Weights that add up to 1

Draw the points `t * a + (1 - t) * b` for `t = 0, 0.1, 0.2, ... 1`. Predict the shape first.

Then try `t = 2` and `t = -1`. Where is the point when `t = 0.5`?

*This leads to: lines, and later to fitting lines to data (Module 13).*

In [ ]:
new_grid(6)
draw_vector(a, label="a")
draw_vector(b, label="b")
for t in np.linspace(0, 1, 11):     # 11 evenly spaced numbers from 0 to 1
    point = t * a + (1 - t) * b
    draw_point(point)
show()

## Deep 4: A third move

Add a third mode, `m = (4, 3)`. Search for all the ways to reach `(7, 4)` using whole numbers from -5 to 5.

With two moves there was one way. Now there are many. **Why?** Look closely at `m`.

*This leads to: getting back home (Module 3).*

In [ ]:
m = np.array([4, 3])
charger = np.array([7, 4])
ways = 0
for c in range(-5, 6):
    for d in range(-5, 6):
        for e in range(-5, 6):
            if np.array_equal(c * a + d * b + e * m, charger):
                print("c =", c, " d =", d, " e =", e)
                ways = ways + 1
print("ways found:", ways)

## Deep 5: What your ML model does inside

Many models take a feature vector, multiply each feature by a weight, and add everything up. That is a dot product, and it is a linear combination of the features.

Here a model scores how good a day is for the park. Change the weights so that rain matters twice as much. Make up a third feature.

In [ ]:
features = np.array([6, 2])        # hours of sun, millimeters of rain
weights = np.array([1.0, -1.5])    # sun is good, rain is bad
score = np.dot(weights, features)
print(score)